# Only sample represenations
# Dependencies

In [ ]:
import numpy as np
import pickle

from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.preprocessing import normalize
from sklearn.mixture import GaussianMixture
from sklearn.metrics import accuracy_score, classification_report, silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


import sys
sys.path.append("/home/rguo_hpc/myfolder/mocap")
from swav.finetune.layers import ProjectionHead, PrototypeLayer
from .utils import PairedEmbeddingDataset, train_prototypes, compute_new_representations

import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# Load
mae_tr  = np.load("/home/rguo_hpc/myfolder/mocap/outputs/fmr1/50/representations/mae_sdannce_tr.npy")[:,25:4525] # 360, 4500, 192
mae_val = np.load("/home/rguo_hpc/myfolder/mocap/outputs/fmr1/50/representations/mae_sdannce_val.npy")[:,25:4525]
mae_feats = np.concatenate([mae_tr, mae_val]) # 480, 4500, 192
num_seq, L, D = mae_feats.shape

In [ ]:
"""
mae_feats = mae_feats.reshape(-1, 50, D) # Reshape to (B, 50, D)
print(mae_feats.shape)
# Normalize by each subsequence
mean = mae_feats.mean(axis=(0, 1), keepdims=True)
std = mae_feats.std(axis=(0, 1), keepdims=True)
mae_feats_norm = (mae_feats - mean) / (std + 1e-8)
"""
feats = mae_feats.reshape(-1, D) # (2160000, 128)
print(feats.shape)

(2160000, 192)


In [ ]:
"""
# PCA
mae_pca = PCA(n_components=128, random_state=42)
mae_pca_feats = mae_pca.fit_transform(mae_feats_norm)
feats_cumvar = np.cumsum(mae_pca.explained_variance_ratio_)
print(f"Variance explained by PCs: {feats_cumvar[-1]:.1%}")
"""

In [5]:
# Generate 2 views ( or by setting view_invariant?)
view1 = feats[4::9]
view2_selected = []
for start in range(0, len(feats), 9):
    end = min(start + 9, len(feats))
    candidates = np.arange(start, end)
    candidates = candidates[candidates != start + 4]  # exclude 5th element
    view2_selected.append(np.random.choice(candidates))
view2 = feats[view2_selected] 

# shuffle
idx = np.random.permutation(len(view1))
view1_shuffled = view1[idx] # 240000, 192
view2_shuffled = view2[idx]

# Load

In [7]:
K = 64
sample_freq = 9

In [ ]:
# load labels
with open("/home/rguo_hpc/myfolder/data/sdannce/data_fmr1.pkl", 'rb') as file:
    data_fmr1 = pickle.load(file)
fmr1_fold_1 = {"train":[402, 404, 405, 406, 407, 408], "valid": [401, 403]} # In total 8 mice, each 3 sequnces with 90000 frames

hlac_labels = []
for mouse in fmr1_fold_1["train"]+fmr1_fold_1["valid"]:
    num_seq = len(data_fmr1[mouse]["hlac"])
    for i in range(num_seq):
        #data_fmr1[401]["ratgen"] [1,1,1]
        hlac = np.squeeze(data_fmr1[mouse]["hlac"][i])
        hlac_labels.append(hlac)

hlac_labels = np.array(hlac_labels).reshape(-1)[::sample_freq]
l_tr = 180000
hlac_tr = hlac_labels[:l_tr]
hlac_val = hlac_labels[l_tr:]
print(hlac_tr.shape)
print(hlac_val.shape)

(180000,)
(60000,)


In [ ]:
"""
# K-means for init
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
kmeans.fit(gmm.means_)
centers = kmeans.cluster_centers_
"""
# GMM for init when projection head is None
gmm = GaussianMixture(n_components=K, covariance_type="diag", random_state=42, n_init=3)
gmm.fit(feats[::9])
centers = gmm.means_

In [ ]:
NUM_PROTOTYPES = K         # <-- match your expected number of behaviors
NUM_EPOCHS = 20
BATCH_SIZE = 1024
PROJECTION_HIDDEN_DIM = 192
PROJECTION_OUT_DIM = 192
lr = 2e-4
weight_decay = 5e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBED_DIM = 192

prototypes, projection_head, dataset = train_prototypes(
                    "view1_shuffled.npy", "view2_shuffled.npy", num_prototypes=NUM_PROTOTYPES, 
                    batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS, 
                    projection_hidden_dim = PROJECTION_HIDDEN_DIM, projection_out_dim=PROJECTION_OUT_DIM,
                    lr = lr, weight_decay=weight_decay) # gmm_means = centers

torch.save(prototypes.state_dict(), "trained_prototypes.pt")
if projection_head is not None:
    torch.save(projection_head.state_dict(), "trained_projection_head.pt")
    print("Saved trained_projection_head.pt")

epoch   0  step    0  loss 4.6793
epoch   0  step  100  loss 7.1629
epoch   0  step  200  loss 7.3352
epoch   0 done, avg loss 7.1765
epoch   1  step    0  loss 7.1933
epoch   1  step  100  loss 0.7688
epoch   1  step  200  loss 0.6741
epoch   1 done, avg loss 1.2396
epoch   2  step    0  loss 0.7466
epoch   2  step  100  loss 0.6746
epoch   2  step  200  loss 0.6854
epoch   2 done, avg loss 0.6792
epoch   3  step    0  loss 0.6974
epoch   3  step  100  loss 0.6785
epoch   3  step  200  loss 0.6538
epoch   3 done, avg loss 0.6453
epoch   4  step    0  loss 0.5696
epoch   4  step  100  loss 0.5926
epoch   4  step  200  loss 0.6110
epoch   4 done, avg loss 0.6249
epoch   5  step    0  loss 0.6287
epoch   5  step  100  loss 0.6866
epoch   5  step  200  loss 0.5543
epoch   5 done, avg loss 0.6098
epoch   6  step    0  loss 0.5927
epoch   6  step  100  loss 0.5875
epoch   6  step  200  loss 0.7194
epoch   6 done, avg loss 0.6004
epoch   7  step    0  loss 0.6288
epoch   7  step  100  loss 0

In [13]:
# After training
# Load prototypes model
prototypes =  PrototypeLayer(embed_dim=EMBED_DIM, num_prototypes=NUM_PROTOTYPES)
checkpoint_model = torch.load("trained_prototypes.pt", map_location=DEVICE, weights_only=False)
prototypes.load_state_dict(checkpoint_model, strict=True)

<All keys matched successfully>

In [ ]:
which = "projection" if PROJECTION_OUT_DIM is not None else "cluster"
# Load projection head if exists
projection_head =  ProjectionHead(in_dim=EMBED_DIM, hidden_dim=PROJECTION_HIDDEN_DIM, out_dim=PROJECTION_OUT_DIM)
checkpoint = torch.load("trained_projection_head.pt", map_location=DEVICE, weights_only=False)
projection_head.load_state_dict(checkpoint, strict=True)

<All keys matched successfully>

In [15]:
new_repr = compute_new_representations(prototypes, torch.from_numpy(feats[::sample_freq]).float(), DEVICE, 
                                       projection_head = projection_head, which="projection",
                                       #projection_head = None, which="cluster"
                                      )
mae_feats_tr = new_repr[:180000,]
mae_feats_val = new_repr[180000:,]

# If you'd rather keep the original D-dim embedding AND add the cluster representation (rather than replace it), concatenate instead:
# combined = torch.cat([F.normalize(dataset.z_a, dim=1), new_repr], dim=1)

In [16]:
# model
model = LogisticRegression(max_iter=500, multi_class='multinomial')
#model = RandomForestClassifier()
# fit & predict
model.fit(mae_feats_tr,  hlac_tr)

y_pred = model.predict(mae_feats_val)
print("Accuracy:", accuracy_score(hlac_val, y_pred))
print("\nClassification Report:\n", classification_report(hlac_val, y_pred))

/home/rguo_hpc/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.7133666666666667

Classification Report:
               precision    recall  f1-score   support

           1       0.70      0.61      0.65     11792
           2       0.64      0.68      0.66     16949
           3       0.75      0.84      0.79      2183
           4       0.75      0.64      0.69      2158
           5       0.45      0.27      0.33       961
           6       0.87      0.95      0.91      6165
           7       0.70      0.72      0.71     12772
           8       0.78      0.81      0.79      7020

    accuracy                           0.71     60000
   macro avg       0.70      0.69      0.69     60000
weighted avg       0.71      0.71      0.71     60000

